# Meta-NATH VisA Phase 3 Workflow

This notebook runs the separate VisA conservative Phase 3 workflow: warmup, before evaluation, Phase 3 consolidation, after evaluation, and acceptance. Mount or copy the VisA dataset to `data/visa` before running the final cell.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

REPO = Path('/kaggle/working/NestedLearningForCAD')
CONFIG = 'conf/visa_phase3.yaml'
SCRIPT = 'scripts/run_server_visa.sh'
VISA_ROOT = REPO / 'data/visa'
MAX_TASKS = ''  # empty means all VisA classes; set to '3' for a shorter smoke run
RUN_PHASE3 = '1'
REQUIRE_ACCEPTED = '0'
STEP_TIMEOUT_SECONDS = '7200'
METANATH_REQUIRE_HF_BACKBONE = '1'

print('repo:', REPO)
print('config:', CONFIG)
print('script:', SCRIPT)
print('visa_root:', VISA_ROOT)
assert REPO.exists(), f'Repo not found: {REPO}'
assert (REPO / CONFIG).exists(), f'Config not found: {CONFIG}'
assert (REPO / SCRIPT).exists(), f'Script not found: {SCRIPT}'


In [ ]:
def run(cmd, cwd=REPO, env=None, timeout=None):
    print('$', ' '.join(map(str, cmd)), flush=True)
    start = time.time()
    proc = subprocess.Popen(
        list(map(str, cmd)),
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='', flush=True)
        rc = proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        raise RuntimeError(f'Command timed out after {timeout}s: {cmd}')
    elapsed = int(time.time() - start)
    print(f'\n[done] exit={rc} elapsed={elapsed}s', flush=True)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


In [ ]:
run([
    sys.executable,
    '-m',
    'py_compile',
    'training/run_experiment.py',
    'dataset/load_dataset.py',
    'scripts/pipeline/run_phase3_consolidation.py',
    'scripts/pipeline/evaluate_checkpoint.py',
    'scripts/pipeline/phase3_acceptance.py',
], cwd=REPO)
run(['bash', '-n', SCRIPT, 'scripts/workflows/run_server_visa.sh'], cwd=REPO)


In [ ]:
if not VISA_ROOT.exists():
    raise FileNotFoundError(
        f'VisA dataset not found at {VISA_ROOT}. Mount/copy VisA there, or edit conf/visa_phase3.yaml dataset.root_dir.'
    )

env = os.environ.copy()
env.update({
    'PYTHON_BIN': sys.executable,
    'CONFIG': CONFIG,
    'PROGRESS': '1',
    'RUN_PHASE3': RUN_PHASE3,
    'REQUIRE_ACCEPTED': REQUIRE_ACCEPTED,
    'STEP_TIMEOUT_SECONDS': STEP_TIMEOUT_SECONDS,
    'HF_HUB_DISABLE_XET': '1',
    'METANATH_REQUIRE_HF_BACKBONE': METANATH_REQUIRE_HF_BACKBONE,
})
if MAX_TASKS:
    env['MAX_TASKS'] = str(MAX_TASKS)

run(['bash', SCRIPT], cwd=REPO, env=env)


In [ ]:
import json

reports = sorted(REPO.glob('results/*visa_phase3*/acceptance_report.json'), key=lambda p: p.stat().st_mtime)
summaries = sorted(REPO.glob('results/*visa_phase3*/phase3_summary.json'), key=lambda p: p.stat().st_mtime)

if reports:
    report_path = reports[-1]
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('latest acceptance:', report_path.relative_to(REPO))
    print('decision:', report.get('decision'), 'accepted:', report.get('accepted'))
    print(json.dumps(report.get('checks', []), indent=2))
else:
    print('No VisA Phase 3 acceptance report found yet.')

if summaries:
    summary_path = summaries[-1]
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('\nlatest phase3 summary:', summary_path.relative_to(REPO))
    print(json.dumps(summary, indent=2)[:3000])
